# Customer Churn Prediction — Machine Learning Mini Project

**Platform:** Google Colab  
**Category:** Supervised Learning (Binary Classification)  
**Dataset:** IBM Telco Customer Churn sample dataset

## Problem statement
Telecommunications companies can lose revenue when customers leave their service. In this project, we use historical customer and service information to predict whether a customer is likely to churn (`Yes`) or stay (`No`).

## Objectives
1. Load and understand a public customer-churn dataset.
2. Clean the data and perform exploratory data analysis (EDA).
3. Engineer a useful feature and prepare data for machine learning.
4. Train and compare Logistic Regression, Decision Tree, Random Forest, and AdaBoost.
5. Evaluate models using cross-validation and evaluate the selected model on a held-out test set.
6. Identify useful churn-related patterns and discuss limitations.

> **Important:** This is an educational project using a fictional sample dataset. Predictions are not guaranteed to generalize to a real company. Churn association does not prove causation.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
import joblib

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
print("Libraries imported successfully.")

## 2. Load the dataset

The CSV is hosted in IBM's public GitHub repository, so Colab can load it directly. If the URL is temporarily unavailable, download the CSV from the [IBM repository](https://github.com/IBM/telco-customer-churn-on-icp4d/blob/master/data/Telco-Customer-Churn.csv) and upload it to Colab, then replace the URL below with the uploaded file path.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_URL)
print(f"Dataset loaded: {df.shape[0]:,} rows and {df.shape[1]} columns")
display(df.head())

## 3. Understand the dataset

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types and non-null counts:")
df.info()

In [ ]:
display(df.describe(include="all").T)

## 4. Check data quality

In [ ]:
print("Missing values by column (before cleaning):")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count").head(10))

print("Exact duplicate rows:", df.duplicated().sum())
if "customerID" in df.columns:
    print("Duplicate customer IDs:", df["customerID"].duplicated().sum())

print("\nTarget distribution:")
display(df["Churn"].value_counts(dropna=False).to_frame("customers"))

## 5. Clean the data and define the target

- `TotalCharges` is stored as text in the source file. Convert it to numeric; blank strings become missing values.
- `customerID` is an identifier, not a useful predictive feature, so it will be excluded from model training.
- Encode `Churn`: `Yes = 1` and `No = 0`.

Missing feature values will be imputed **inside the machine-learning pipeline**, so the imputer learns from training folds only.

In [ ]:
df = df.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

# Remove rows where the target is missing or could not be encoded.
df = df.dropna(subset=["Churn"]).copy()
df["Churn"] = df["Churn"].astype(int)

print("Shape after basic cleaning:", df.shape)
print("Missing TotalCharges values:", df["TotalCharges"].isna().sum())
print("Target values:", sorted(df["Churn"].unique()))
display(df.head())

## 6. Exploratory Data Analysis (EDA)

### 6.1 Overall churn distribution

In [ ]:
churn_counts = df["Churn"].map({0: "No churn", 1: "Churn"}).value_counts()
ax = sns.barplot(x=churn_counts.index, y=churn_counts.values)
ax.set(title="Customer Churn Distribution", xlabel="Customer outcome", ylabel="Number of customers")
plt.show()

churn_rate = df["Churn"].mean() * 100
print(f"Overall churn rate: {churn_rate:.2f}%")

### 6.2 Churn by contract type

In [ ]:
contract_churn = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100
contract_churn = contract_churn.rename(columns={0: "No churn", 1: "Churn"})
display(contract_churn.round(2))

plot_df = df.assign(ChurnLabel=df["Churn"].map({0: "No churn", 1: "Churn"}))
plt.figure(figsize=(8, 5))
sns.countplot(data=plot_df, x="Contract", hue="ChurnLabel")
plt.title("Churn by Contract Type")
plt.xlabel("Contract")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

### 6.3 Tenure and monthly charges

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=df, x="tenure", hue="Churn", bins=24, element="step", stat="density", common_norm=False, ax=axes[0])
axes[0].set_title("Tenure Distribution by Churn")
axes[0].set_xlabel("Tenure (months)")

sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1])
axes[1].set_title("Monthly Charges by Churn")
axes[1].set_xlabel("Churn (0 = No, 1 = Yes)")
plt.tight_layout()
plt.show()

### 6.4 Churn by internet service

In [ ]:
internet_churn = pd.crosstab(df["InternetService"], df["Churn"], normalize="index") * 100
display(internet_churn.rename(columns={0: "No churn (%)", 1: "Churn (%)"}).round(2))

internet_churn.rename(columns={0: "No churn", 1: "Churn"}).plot(kind="bar", figsize=(8, 4))
plt.title("Churn Proportion by Internet Service")
plt.ylabel("Proportion within service group")
plt.xlabel("Internet service")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### EDA observations

Use the charts and displayed tables to write 2–4 observations after running the notebook. For example, inspect whether churn differs by contract type and tenure. Do not assume a pattern before seeing the actual plots, and remember that these relationships are associations, not proof of cause.

## 7. Feature engineering

We create `NumAdditionalServices`, which counts the customer's add-on services marked `Yes`. This is a simple derived feature based on existing service columns. We keep the original features as well so the model can learn their individual effects.

In [ ]:
service_columns = [
    "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]

# Count the selected add-on services that are marked "Yes" for each customer.
df["NumAdditionalServices"] = df[service_columns].eq("Yes").sum(axis=1)

display(df[[*service_columns, "NumAdditionalServices", "Churn"]].head())
print("Created feature: NumAdditionalServices")

## 8. Define features and split the data

We use a stratified 80/20 split. Stratification keeps the churn/non-churn proportion similar in the training and test sets. The test set is kept out of model fitting and hyperparameter selection.

In [ ]:
TARGET = "Churn"
DROP_COLUMNS = [TARGET, "customerID"]
X = df.drop(columns=[c for c in DROP_COLUMNS if c in df.columns])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Training churn rate: {:.2f}%".format(y_train.mean() * 100))
print("Testing churn rate:  {:.2f}%".format(y_test.mean() * 100))

## 9. Build preprocessing pipelines

Numeric features are median-imputed and standardized. Categorical features are imputed with their most frequent training value and one-hot encoded. Putting preprocessing inside a pipeline helps prevent data leakage during cross-validation.

In [ ]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number", "bool"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Preprocessing setup complete.")

## 10. Train and compare four supervised algorithms

We use stratified cross-validation on the training data to compare models. F1-score is the primary comparison metric because the dataset has fewer churners than non-churners; precision, recall and ROC-AUC are also reported. `class_weight='balanced'` is used for Logistic Regression, Decision Tree and Random Forest to help account for class imbalance.

In [ ]:
model_classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=150, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=RANDOM_STATE)
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = []
model_pipelines = {}

for name, classifier in model_classifiers.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", classifier)])
    model_pipelines[name] = pipe
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    cv_results.append({
        "Model": name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1": scores["test_f1"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean(),
        "F1 Std": scores["test_f1"].std()
    })

cv_comparison = pd.DataFrame(cv_results).sort_values("CV F1", ascending=False).reset_index(drop=True)
display(cv_comparison.style.format({c: "{:.3f}" for c in cv_comparison.columns if c != "Model"}))

## 11. Hyperparameter tuning for Random Forest

GridSearchCV tests a small set of Random Forest settings using only the training data. It selects settings by mean cross-validation F1-score; the held-out test set is not used for tuning.

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1
    ))
])

param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 10],
    "classifier__min_samples_leaf": [1, 3]
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=1,
    refit=True
)
grid_search.fit(X_train, y_train)

print("Best Random Forest CV F1:", round(grid_search.best_score_, 4))
print("Best parameters:")
print(grid_search.best_params_)

## 12. Select the model using training cross-validation

We choose between the four baseline pipelines and the tuned Random Forest using cross-validation F1 on the training set. The test set is used only after this choice for the final evaluation.

In [ ]:
candidate_scores = cv_comparison[["Model", "CV F1"]].copy()
tuned_rf_score = pd.DataFrame([{"Model": "Tuned Random Forest", "CV F1": grid_search.best_score_}])
candidate_scores = pd.concat([candidate_scores, tuned_rf_score], ignore_index=True)
candidate_scores = candidate_scores.sort_values("CV F1", ascending=False).reset_index(drop=True)
display(candidate_scores.style.format({"CV F1": "{:.3f}"}))

best_name = candidate_scores.loc[0, "Model"]
if best_name == "Tuned Random Forest":
    best_model = grid_search.best_estimator_
else:
    best_model = model_pipelines[best_name]
    best_model.fit(X_train, y_train)

print("Selected model:", best_name)

## 13. Final evaluation on the held-out test set

The following metrics are calculated from predictions on records that were not used to fit or tune the selected model. Recall measures the proportion of actual churners detected; precision measures how many customers flagged as churners actually churned.

In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

final_metrics = pd.DataFrame([{
    "Selected model": best_name,
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_proba)
}])
display(final_metrics.style.format({c: "{:.3f}" for c in final_metrics.columns if c != "Selected model"}))

print("Classification report:")
print(classification_report(y_test, y_pred, target_names=["No churn", "Churn"], zero_division=0))

### 13.1 Confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["No churn", "Churn"],
    cmap="Blues",
    values_format="d"
)
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

### 13.2 ROC curve

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title(f"ROC Curve — {best_name}")
plt.tight_layout()
plt.show()

## 14. Interpret the selected model

For tree-based models, we display impurity-based feature importance. For Logistic Regression, we display the largest absolute coefficients. One-hot encoded columns are labeled using the fitted preprocessing pipeline.

In [ ]:
fitted_preprocessor = best_model.named_steps["preprocessor"]
fitted_classifier = best_model.named_steps["classifier"]
feature_names = fitted_preprocessor.get_feature_names_out()

if hasattr(fitted_classifier, "feature_importances_"):
    importances = fitted_classifier.feature_importances_
    importance_label = "Feature importance"
elif hasattr(fitted_classifier, "coef_"):
    importances = np.abs(fitted_classifier.coef_[0])
    importance_label = "Absolute coefficient magnitude"
else:
    importances = None
    importance_label = None

if importances is not None:
    feature_importance = pd.DataFrame({
        "Feature": feature_names,
        importance_label: importances
    }).sort_values(importance_label, ascending=False).head(15)
    display(feature_importance)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=feature_importance, x=importance_label, y="Feature")
    plt.title(f"Top 15 Features — {best_name}")
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance is not available for this estimator.")

## 15. Save the selected model

The saved pipeline includes both preprocessing and the trained classifier. This means future data must be supplied with the same raw feature columns used in `X`.

In [ ]:
MODEL_PATH = "customer_churn_model.joblib"
joblib.dump(best_model, MODEL_PATH)
print(f"Saved model pipeline to: {MODEL_PATH}")

# Download the trained pipeline from Colab to your computer.
from google.colab import files
files.download(MODEL_PATH)

## 16. Results and conclusion — complete after running

Fill in these points using the real outputs above (do not copy assumed scores):

1. **Best cross-validation model:** Which model had the highest CV F1-score?
2. **Final test performance:** Record Accuracy, Precision, Recall, F1-score, and ROC-AUC for the selected model.
3. **Churn patterns:** Write 2–3 observations from the EDA charts (for example, contract type or tenure groups).
4. **Most influential features:** Name the leading features shown in the model interpretation chart, and explain them cautiously as associations.
5. **Limitations:** The dataset is a sample snapshot; it may not reflect current customers or other telecom companies. The selected decision threshold is the model's default threshold, and false positives/false negatives have different business costs.
6. **Future scope:** Test alternative thresholds, examine calibration, validate on newer data, or investigate how retention strategies affect outcomes.

### Project workflow summary
**Data collection → Data cleaning → EDA → Feature engineering → Train/test split → Preprocessing pipeline → Cross-validation → Hyperparameter tuning → Final test evaluation → Feature interpretation → Saved model**